# Parquet Data Dictionary Builder

This notebook builds an Excel data dictionary for the three core processed parquet files:

- `consumer_master.parquet`
- `vend.parquet`
- `consumption.parquet`

The output workbook is designed to be practical for review and handoff. It includes:

- a dataset summary sheet
- one detailed dictionary sheet per parquet file
- a combined dictionary sheet across all datasets

The notebook reads directly from the processed parquet files, so it works without the raw CSVs.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
from pandas.api.types import (
    is_bool_dtype,
    is_datetime64_any_dtype,
    is_integer_dtype,
    is_float_dtype,
    is_numeric_dtype,
)


def find_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").exists() and (candidate / "config").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root containing src/ and config/.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: F:\Secure\CashFlowMgmt


In [2]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "parquet_data_dictionary.xlsx"

PARQUET_FILES = {
    "consumer_master": PROCESSED_DIR / "consumer_master.parquet",
    "vend": PROCESSED_DIR / "vend.parquet",
    "consumption": PROCESSED_DIR / "consumption.parquet",
}

for dataset_name, path in PARQUET_FILES.items():
    print(dataset_name, "exists" if path.exists() else "missing", path)

print(f"Excel output will be written to: {OUTPUT_PATH}")

consumer_master exists F:\Secure\CashFlowMgmt\data\processed\consumer_master.parquet
vend exists F:\Secure\CashFlowMgmt\data\processed\vend.parquet
consumption exists F:\Secure\CashFlowMgmt\data\processed\consumption.parquet
Excel output will be written to: F:\Secure\CashFlowMgmt\data\processed\parquet_data_dictionary.xlsx


In [3]:
# Load the three processed datasets.
datasets = {name: pd.read_parquet(path) for name, path in PARQUET_FILES.items()}

display(pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "path": str(PARQUET_FILES[name]),
    }
    for name, df in datasets.items()
]))

,dataset,rows,columns,path
0,consumer_master,1548361,38,F:\Secure\CashFlowMgmt\data\processed\consumer...
1,vend,1503445,19,F:\Secure\CashFlowMgmt\data\processed\vend.par...
2,consumption,580860,25,F:\Secure\CashFlowMgmt\data\processed\consumpt...


## Profiling helpers

These helpers create a practical data dictionary rather than just listing column names. The output includes completeness, distinct counts, example values, and basic min/max profiling where it is safe to do so.

In [4]:
def classify_semantic_type(series: pd.Series) -> str:
    if is_datetime64_any_dtype(series):
        return "datetime"
    if is_bool_dtype(series):
        return "boolean"
    if is_integer_dtype(series):
        return "integer"
    if is_float_dtype(series):
        return "float"
    if is_numeric_dtype(series):
        return "numeric"
    return "string_or_mixed"


def safe_examples(series: pd.Series, limit: int = 5) -> str:
    examples = series.dropna().astype("string").str.strip()
    examples = examples[examples.ne("")].drop_duplicates().head(limit).tolist()
    return " | ".join(examples) if examples else ""


def safe_min_value(series: pd.Series):
    cleaned = series.dropna()
    if cleaned.empty:
        return pd.NA
    try:
        return cleaned.min()
    except TypeError:
        return pd.NA


def safe_max_value(series: pd.Series):
    cleaned = series.dropna()
    if cleaned.empty:
        return pd.NA
    try:
        return cleaned.max()
    except TypeError:
        return pd.NA


def key_role(column_name: str) -> str:
    if column_name in {"consumernumber", "meterno", "consumernumber_normalized", "meterno_normalized"}:
        return "business_key"
    if column_name in {"consumer_master_id", "resolved_master_id"}:
        return "internal_key"
    if column_name in {"midnightdate", "midnightdate_parsed", "date", "issuedate", "issuedate_parsed", "vend_date", "balanceupdatedon", "balanceupdatedon_parsed", "meterinstallationdate", "meterinstallationdate_parsed"}:
        return "date_or_time"
    return "attribute"


def build_dataset_summary(name: str, df: pd.DataFrame, path: Path) -> dict:
    return {
        "dataset": name,
        "file_name": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024 ** 2), 2),
        "unique_consumernumber_normalized": int(df["consumernumber_normalized"].nunique(dropna=True)) if "consumernumber_normalized" in df.columns else pd.NA,
        "unique_meterno_normalized": int(df["meterno_normalized"].nunique(dropna=True)) if "meterno_normalized" in df.columns else pd.NA,
        "path": str(path),
    }


def build_data_dictionary(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    rows = []
    row_count = len(df)

    for position, column in enumerate(df.columns, start=1):
        series = df[column]
        non_null = int(series.notna().sum())
        missing = int(series.isna().sum())
        distinct = int(series.dropna().nunique())
        semantic_type = classify_semantic_type(series)

        if semantic_type in {"datetime", "integer", "float", "numeric", "boolean"}:
            min_value = safe_min_value(series)
            max_value = safe_max_value(series)
        else:
            min_value = pd.NA
            max_value = pd.NA

        rows.append({
            "dataset": dataset_name,
            "column_position": position,
            "column_name": column,
            "pandas_dtype": str(series.dtype),
            "semantic_type": semantic_type,
            "role_hint": key_role(column),
            "row_count": row_count,
            "non_null_count": non_null,
            "missing_count": missing,
            "missing_pct": round(missing / row_count * 100, 2) if row_count else 0.0,
            "distinct_non_null_count": distinct,
            "distinct_pct_of_rows": round(distinct / row_count * 100, 2) if row_count else 0.0,
            "sample_values": safe_examples(series),
            "min_value": min_value,
            "max_value": max_value,
            "notes": "",
        })

    dictionary_df = pd.DataFrame(rows)
    return dictionary_df


dataset_summary_df = pd.DataFrame([
    build_dataset_summary(name, df, PARQUET_FILES[name])
    for name, df in datasets.items()
])

dictionary_frames = {name: build_data_dictionary(df, name) for name, df in datasets.items()}
combined_dictionary_df = pd.concat(dictionary_frames.values(), ignore_index=True)

In [5]:
display(dataset_summary_df)
display(dictionary_frames["consumer_master"].head(10))

,dataset,file_name,rows,columns,memory_mb,unique_consumernumber_normalized,unique_meterno_normalized,path
0,consumer_master,consumer_master.parquet,1548361,38,2626.84,1548361,987408,F:\Secure\CashFlowMgmt\data\processed\consumer...
1,vend,vend.parquet,1503445,19,1061.59,639662,639759,F:\Secure\CashFlowMgmt\data\processed\vend.par...
2,consumption,consumption.parquet,580860,25,595.43,19270,19270,F:\Secure\CashFlowMgmt\data\processed\consumpt...


,dataset,column_position,column_name,pandas_dtype,semantic_type,role_hint,row_count,non_null_count,missing_count,missing_pct,distinct_non_null_count,distinct_pct_of_rows,sample_values,min_value,max_value,notes
0,consumer_master,1,consumername,string,string_or_mixed,attribute,1548361,987434,560927,36.23,347771,22.46,MUKESH MANDASL | VIJAY BHUSHAN SAH | NITISH KU...,<NA>,<NA>,
1,consumer_master,2,consumernumber,string,string_or_mixed,business_key,1548361,1548361,0,0.00,1548361,100.00,1231053137 | 122410808173 | 123111331382 | 120...,<NA>,<NA>,
2,consumer_master,3,meterno,string,string_or_mixed,business_key,1548361,987433,560928,36.23,987408,63.77,NB11887278 | NB11916418 | NB11525680 | NB10837...,<NA>,<NA>,
3,consumer_master,4,feedercode,string,string_or_mixed,attribute,1548361,705284,843077,54.45,427,0.03,FD426 | FD093 | FD01565 | FD01504 | FD244,<NA>,<NA>,
4,consumer_master,5,dtcode,string,string_or_mixed,attribute,1548361,705121,843240,54.46,11679,0.75,DT09172 | DT01967 | DT13133 | DT39648 | DT05215,<NA>,<NA>,
5,consumer_master,6,metertype,string,string_or_mixed,attribute,1548361,987433,560928,36.23,3,0.00,Single phase | Three phase | LT CT,<NA>,<NA>,
6,consumer_master,7,sanctionedload,string,string_or_mixed,attribute,1548361,1548360,1,0.00,460,0.03,0.25 | 1.0 | 1.0 kW | 0.25 kW | 1.11,<NA>,<NA>,
7,consumer_master,8,privilagedconsumer,string,string_or_mixed,attribute,1548361,987433,560928,36.23,2,0.00,No | Yes,<NA>,<NA>,
8,consumer_master,9,netmetering,string,string_or_mixed,attribute,1548361,987433,560928,36.23,2,0.00,No | Yes,<NA>,<NA>,
9,consumer_master,10,tariffcode,string,string_or_mixed,attribute,1548361,987433,560928,36.23,13,0.00,KJ | DS1D | DS2D | NDS1D | IAS1,<NA>,<NA>,


## Write the Excel workbook

This cell writes the workbook to `data/processed/parquet_data_dictionary.xlsx`. If the file already exists, it is overwritten.

In [6]:
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    dataset_summary_df.to_excel(writer, sheet_name="dataset_summary", index=False)
    combined_dictionary_df.to_excel(writer, sheet_name="all_columns", index=False)

    for dataset_name, dictionary_df in dictionary_frames.items():
        sheet_name = dataset_name[:31]
        dictionary_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Data dictionary workbook written to: {OUTPUT_PATH}")

Data dictionary workbook written to: F:\Secure\CashFlowMgmt\data\processed\parquet_data_dictionary.xlsx


## Optional next steps

Good follow-ups if you want to extend this workbook later:

- add business definitions to the `notes` column
- flag suspected key fields and join-critical fields more explicitly
- add source-to-source field overlap tables
- add data-quality warnings for high-missing or low-coverage columns